# Examples of machine learning models

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression, make_blobs
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split

In [ ]:
random_state = 1234

## Classification
Let's go back to the stroke data.

In [ ]:
# load the data and take a peek
df = pd.read_csv("../01_stroke/stroke_data.csv")
# Keep the numeric imaging predictors
VC_preds = ["CALCVol", "CALCVolProp", "MATXVol", "MATXVolProp", "LRNCVol", 
    "LRNCVolProp", "MaxCALCArea", "MaxCALCAreaProp", "MaxDilationByArea", 
    "MaxMATXArea", "MaxMATXAreaProp", "MaxLRNCArea", "MaxLRNCAreaProp", 
    "MaxMaxWallThickness", "MaxRemodelingRatio", "MaxStenosisByArea", 
    "MaxWallArea", "WallVol", "MaxStenosisByDiameter"]
X_train, X_test, y_train, y = train_test_split(df[VC_preds], df["Stroke"] == "Y", random_state=random_state)
print("Training positive percentage:",sum(y_train) / len(y_train))
print("Testing positive percentage:",sum(y)/ len(y))

In [ ]:
tree = DecisionTreeClassifier(random_state=random_state)
tree.fit(X_train, y_train)

In [ ]:
# What's the accuracy?
# data "leakage" - same data to train and to evaluate
tree.score(X_train, y_train)

In [ ]:
# try on test (should actually be val, but I'm being lazy here)
tree.score(X_test, y)

In [ ]:
plt.figure(figsize=(30,20))
plot_tree(
    tree,
    class_names=["Stroke", "No Stroke"],
    feature_names=VC_preds,
    impurity=False,
    filled=True,
    fontsize=14
)
plt.show()

In [ ]:
# Try again, restrict the tree depth
tree = DecisionTreeClassifier(random_state=random_state, max_depth=3)
tree.fit(X_train, y_train)
print(tree.score(X_train, y_train))
plt.figure(figsize=(20,12))
plot_tree(
    tree,
    class_names=["Stroke", "No Stroke"],
    feature_names=VC_preds,
    impurity=False,
    filled=True,
    fontsize=14
)
plt.show()

In [ ]:
# how does it behave on the test now?
tree.score(X_test, y)

In [ ]:
fi = tree.feature_importances_
ticks = np.arange(len(VC_preds))
plt.bar(ticks, fi)
plt.xticks(ticks=ticks, labels=VC_preds, rotation = 90)
plt.show()


## Regression

In [ ]:
# Made-up numbers
X, y = make_regression(n_features=1, n_targets=1, n_samples=100, noise=50, random_state=random_state)
# y += 400 # big impact on MAPE
plt.scatter(X, y)

In [ ]:
X_train, X_test, y_train, y = train_test_split(X, y, random_state=random_state)

# scatter both
plt.scatter(X_train, y_train, label="Training")
plt.scatter(X_test, y, label="Testing")
plt.legend()

In [ ]:
lr = LinearRegression().fit(X_train, y_train)
print(lr.coef_, lr.intercept_)
# add a line
x_line = np.array([-4, 4])
y_line = x_line * lr.coef_ + lr.intercept_

plt.scatter(X_train, y_train, label="Training")
plt.scatter(X_test, y, label="Testing")
plt.plot(x_line, y_line, label="Best fit line", color='r')
plt.legend()

In [ ]:
# Evaluation metrics
def regression_metrics(model, X, y):
    y_pred = model.predict(X)
    mse = ((y_pred - y) ** 2).mean()
    rmse = np.sqrt(mse)
    mae = np.abs(y_pred - y).mean()
    mape = 100*np.abs((y_pred - y) / y).mean()

    print(f"MSE: {mse:.1f}")
    print(f"RMSE: {rmse:.1f}")
    print(f"MAE: {mae:.1f}")
    print(f"MAPE: {mape:.1f}") # not useful here, too many close to zero values
    print(f"R^2: {model.score(X, y):.2f}")

regression_metrics(lr, X_train, y_train)

In [ ]:
tree_reg = DecisionTreeRegressor(max_depth =3).fit(X_train, y_train)
regression_metrics(tree_reg, X_train, y_train)

In [ ]:
plt.scatter(X_train, y_train, label="Training")
plt.scatter(X_test, y, label="Testing")
x_samples = np.linspace(-4,4,100).reshape(100, 1)
plt.plot(x_samples, tree_reg.predict(x_samples), color='k', linestyle='--', label="Decision tree prediction")
plt.legend()

## Clustering